# 01 Probe set
Run 10 probe tasks 5 times on two models and compare.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-01"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Step 2: pick two models

In [ ]:
from kit.llm import call_llm, offline
from kit.prices import cost
import json, re
import pandas as pd

MODEL_A = os.environ.get("PROBE_MODEL_A", "gemini-3.5-flash-lite")
MODEL_B = os.environ.get("PROBE_MODEL_B", "gemini-3.6-flash")
if offline():
    MODEL_A, MODEL_B = "mock-model-a", "mock-model-b"
RUNS = 5
print(MODEL_A, "vs", MODEL_B)

Step 3: load the probes and their pass criteria

The kit's ten probes are examples. Rewrite them in `data/probes.csv` (or in this cell) as ten tasks for your own product's help job. Each row's `check_type` and `check_value` is its pass criterion: write them all before you run anything below.

In [ ]:
probes = pd.read_csv(DATA / "probes.csv").fillna("")
probes

Step 4: define the pass checks

In [ ]:
def passes(text, check_type, check_value):
    if check_type == "must_contain":
        return check_value in text
    if check_type == "regex":
        return re.search(check_value, text.strip(), re.S) is not None
    if check_type == "valid_json":
        try:
            json.loads(text.strip())
            return True
        except ValueError:
            return False
    raise ValueError(f"Unknown check type {check_type}")

print(passes('{"ok": true}', "valid_json", ""), passes("hi", "must_contain", "bye"))

Step 5: run every probe 5 times on each model

In [ ]:
rows = []
for model in [MODEL_A, MODEL_B]:
    for p in probes.itertuples():
        for run in range(1, RUNS + 1):
            r = call_llm(p.task, model=model, temperature=0.7, max_output_tokens=300)
            rows.append({"model": model, "probe_id": p.probe_id, "run": run,
                         "check_type": p.check_type, "check_value": p.check_value,
                         "passed": passes(r["text"], p.check_type, p.check_value),
                         "input_tokens": r["input_tokens"], "output_tokens": r["output_tokens"],
                         "stop_reason": r["stop_reason"],
                         "cost_usd": cost(r["input_tokens"], r["output_tokens"], model),
                         "latency_s": r["latency_s"], "text": r["text"][:300]})
runs = pd.DataFrame(rows)
runs.to_csv(OUTPUT / "probe_runs.csv", index=False)  # Stage 2 reads the failed runs from this file
print(len(runs), "calls, saved to", OUTPUT.name + "/probe_runs.csv")

Step 6: pass rate per probe

In [ ]:
runs.pivot_table(index="probe_id", columns="model", values="passed", aggfunc="mean")

Step 7: compare the two models

In [ ]:
summary = runs.groupby("model").agg(pass_rate=("passed", "mean"), cost_per_task=("cost_usd", "mean"),
                                    median_latency_s=("latency_s", "median"), calls=("passed", "size"))
summary["cost_per_task"] = summary["cost_per_task"].round(6)
summary.to_csv(OUTPUT / "probe_summary.csv")  # Stage 4 scores models with these numbers
summary

Step 8: save the files for later stages

`probe_runs.csv` (every run) and `probe_summary.csv` (the table above) are now in `stage-01/`. In Colab, download both from the Files panel: Stage 2 adds your failed probes to its backlog, and Stage 4 scores models with your numbers. Move them into `stage-01/` when you fork the kit in Chapter 3.

Step 9: five predicted weaknesses

Write five weaknesses you expect your product to have. Tie each one to one layer: model, context, harness or surface. Then pick one, show with the run evidence above that it is false, and rewrite it in a narrower form that matches what the runs show.

| # | Prediction, written before the runs | Layer | What the runs showed | Verdict |
| --- | --- | --- | --- | --- |
| 1 |  |  |  |  |
| 2 |  |  |  |  |
| 3 |  |  |  |  |
| 4 |  |  |  |  |
| 5 |  |  |  |  |

Disproved prediction, rewritten narrower:

Step 10: README

**Decisions:**

**What I measured:**

**What I would do next:**